# MEDILEAF.AI — ResNet50 Transfer Learning, Validation Threshold Calibration & Held-Out OOD Evaluation (Google Colab GPU)

**Academic Project:** Medicinal-Leaf Identifier with a Reject Option (`Computer Vision Course Project | CA-1`)  
**Dataset:** Indian Medicinal Leaves Image Datasets (Version 3) — Mendeley Data (`10.17632/748f8jkphb.3`, CC BY 4.0)  
**Core Stack:** Python · PyTorch · Torchvision ResNet50 · OpenCV · Pillow · Scikit-learn · Matplotlib

> **Strict Integrity Rules:**
> 1. Held-out OOD species are isolated prior to training and NEVER used in training, validation, or threshold calibration.
> 2. Rejection thresholds (MSP & Energy) are calibrated strictly on `validation_manifest.csv`.
> 3. Evaluation transforms in `src/data.py` (`get_eval_transforms()`) are identical to Streamlit inference in `src/inference.py`.

In [ ]:
# 1. Verify GPU availability & clone/mount MEDILEAF.AI workspace
import torch, sys
print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))

In [ ]:
# 2. Install requirements (if running in a fresh Colab runtime)
!pip install -q -r requirements.txt

In [ ]:
# 3. Place/extract Mendeley dataset under data/raw/ and inspect integrity
!python scripts/inspect_dataset.py --raw-dir data/raw
!head -n 20 reports/dataset/dataset_summary.md

In [ ]:
# 4. Create leakage-free species split (Train / Validation / Known Test / Held-Out OOD Test)
!python scripts/prepare_dataset.py --ood-ratio 0.20 --val-ratio 0.15 --test-ratio 0.15 --seed 42
!cat data/processed/split_configuration.json

In [ ]:
# 5. Train & fine-tune ResNet50 (Stage 1: Head warmup -> Stage 2: Selective layer3/layer4/fc fine-tuning)
!python -m src.train --epochs-head 5 --epochs-finetune 20 --batch-size 32 --seed 42 --target-coverage 0.90

In [ ]:
# 6. Evaluate Known Test + Held-Out OOD Species (MSP vs Energy OOD AUROC/AUPR, Coverage, Selective Accuracy)
!python -m src.evaluate --batch-size 32
!cat reports/metrics/evaluation_summary.json

In [ ]:
# 7. Consistency Check: Verify unified MediLeafPredictor matches evaluation preprocessing & rejection rule
import pandas as pd
from src.inference import MediLeafPredictor

predictor = MediLeafPredictor()
print('Predictor ready:', predictor.is_ready)
known_df = pd.read_csv('data/processed/known_test_manifest.csv')
ood_df = pd.read_csv('data/processed/ood_test_manifest.csv')

sample_known = known_df.iloc[0]
print('Known Sample:', sample_known['species'], '->', predictor.predict_file(sample_known['filepath']).to_dict())

sample_ood = ood_df.iloc[0]
print('OOD Sample:', sample_ood['species'], '->', predictor.predict_file(sample_ood['filepath']).to_dict())

In [ ]:
# 8. Package trained artifacts/ & reports/ for download into local Streamlit workspace
!zip -r medileaf_trained_outputs.zip artifacts/ reports/ data/processed/